# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a step-by-step guide for loading and exploring the FAIR² dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL: [https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json](https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json)

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant --quiet

## 1. Data Loading
Load Croissant metadata and instantiate the dataset object.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'
# Load the dataset
dataset = mlc.Dataset(croissant_url)
# Access the metadata object
md = dataset.metadata

print(f"Dataset name: {md.name}\n\nDescription: {md.description}")

## 2. Data Overview
Review available record sets, their fields and columns. All elements are referenced by their `@id` as specified by the Croissant schema.

In [ ]:
# List all record sets by their @id

record_sets = md.record_sets
print(f"Total record sets: {len(record_sets)}\n")
for rs in record_sets:
    print(f"RecordSet @id: {rs.id}")
    print(f"  Name       : {rs.name}")
    print(f"  Description: {getattr(rs, 'description', '')}")
    print(f"  Fields: (by @id)")
    for field in rs.fields:
        print(f"    - {field.id} (name: {field.name}, type: {getattr(field, 'data_type', 'N/A')})")
    print(f"  Columns in tabular resource (by @id):")
    for column in getattr(rs, 'columns', []):
        print(f"    - {column.id} (name: {column.name}, type: {getattr(column, 'data_type', 'N/A')})")
    print("\n")

## 3. Data Extraction
Load the records from a chosen record set into a DataFrame for analysis. Below, we demonstrate this for the main protein abundance/annotation table.

In [ ]:
# --- Identify main record set for extraction ---
# Find protein record set: choose the one that contains 'protein' in its name or fields (customize as needed)
selected_rs = None
for rs in md.record_sets:
    f_names = [field.name.lower() for field in rs.fields]
    if ('protein' in rs.name.lower()) or any('protein' in fn for fn in f_names):
        selected_rs = rs
        break
if not selected_rs:
    selected_rs = md.record_sets[0]  # Fallback: pick first

# Display chosen record set @id
record_set_id = selected_rs.id
print(f"Chosen RecordSet by @id: {record_set_id} (name: {selected_rs.name})\n")

# Get list of all record set ids
record_set_ids = [rs.id for rs in md.record_sets]

dataframes = {}
for rs_id in record_set_ids:
    # Each record is a dictionary where keys are field @ids
    records = list(dataset.records(record_set=rs_id))
    if records:
        df = pd.DataFrame(records)
        dataframes[rs_id] = df
        print(f"Loaded DataFrame for {rs_id} with shape {df.shape}")
    else:
        print(f"No records found for {rs_id}")

# Show the first few columns/rows from the main record set
print(f"\nColumns for {record_set_id}:")
print(dataframes[record_set_id].columns.tolist())
dataframes[record_set_id].head()

## 4. Exploratory Data Analysis (EDA)
Explore and process the tabular data, using only field/column `@id`s, for example to filter, normalize, and group.

In [ ]:
# Find a numeric field @id for analysis (e.g. the one with 'abundance', 'mw', 'peptide', 'coverage' in its name)
rs_fields = [field for field in selected_rs.fields]

# Heuristically pick a field containing 'abundance' or 'mw' in its name (use only @id)
numeric_field_id = None
for f in rs_fields:
    if any(x in f.name.lower() for x in ['abundance', 'mw', 'molecular_weight', 'coverage', 'count']):
        numeric_field_id = f.id
        break

if not numeric_field_id:
    # Fallback: pick first field
    numeric_field_id = rs_fields[0].id

print(f"Using numeric field: {numeric_field_id}\n")

# Filter (after making sure column exists and is numeric)
main_df = dataframes[record_set_id].copy()
if numeric_field_id in main_df.columns:
    # Try to ensure numeric values
    main_df[numeric_field_id] = pd.to_numeric(main_df[numeric_field_id], errors='coerce')
    threshold = main_df[numeric_field_id].mean(skipna=True)
    filtered_df = main_df[main_df[numeric_field_id] > threshold]
    print(f"Filtered records with {numeric_field_id} > mean ({threshold:.2f}): {len(filtered_df)} records")

    # Normalize
    filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean())/filtered_df[numeric_field_id].std()
    print(f"First 5 normalized records for {numeric_field_id}:")
    display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())
else:
    print(f"Field {numeric_field_id} not found in columns.")

# Optionally group by a categorical field (e.g. one with 'description' or 'sample')
group_field_id = None
for f in rs_fields:
    if any(x in f.name.lower() for x in ['description', 'sample', 'accession']):
        group_field_id = f.id
        break

if group_field_id and group_field_id in filtered_df:
    grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
    print(f"Grouped {numeric_field_id} by {group_field_id} (mean values):")
    display(grouped_df.head())
else:
    print("No suitable group field found or not present in DataFrame.")

## 5. Visualization
Visualize the distribution of the main numeric field (`@id`) from the main record set.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Check if the field is present, plot its distribution
if numeric_field_id in main_df:
    plt.figure(figsize=(8,5))
    sns.histplot(main_df[numeric_field_id].dropna(), bins=30, kde=True)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel('Frequency')
    plt.show()
else:
    print("Numeric field not found: cannot display histogram.")

## 6. Conclusion
In this notebook, we used the `mlcroissant` library to load, inspect, filter, process, and visualize a FAIR²-compliant mass spectrometry protein dataset. Operations were referenced by Croissant `@id`, ensuring schema-compliant, reproducible workflows.

For more information about the schema or advanced data integration/processing, refer to [mlcroissant documentation](https://mlcommons.org/croissant/spec/).